In [7]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.13.12 environment at: /home/mohamed-tamer/Downloads/MyProjects/LLMops/.venv
Checked 6 packages in 8ms


In [5]:
import os
from dotenv import load_dotenv

# .env lives in the project root; the notebook runs from its own directory
load_dotenv("../.env", override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is missing from ../.env")

In [8]:
from langchain_community.document_loaders import TextLoader

/tmp/ipykernel_204022/2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


### Data ingestion and preprocessing


In [11]:
loader = TextLoader("../data/Agentic AI.txt", encoding="utf-8")
documents = loader.load()

In [14]:
documents[0].page_content[:500]  # show the first 500 characters of the first document

'Understanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals.  This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to '

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

In [20]:
text_chunks = text_splitter.split_documents(documents)

In [21]:
text_chunks

[Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Understanding Agentic AI\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals.  This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\nKey Characteristics of Agentic AI\nAgentic AI systems are distinct from traditional AI models due to several core characteristics:\n* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.\n* Autonomy: They can operate independently, making decisions and taking actions without continuous human oversight.\n* Perception: They can interpret information from their environment to inform their decisions.\n* Planning and Reasoning: They can formulate plans to reach their goals and reason about the con

In [22]:
! uv pip install faiss-cpu

Using Python 3.13.12 environment at: /home/mohamed-tamer/Downloads/MyProjects/LLMops/.venv
Resolved 3 packages in 677ms                                         
Prepared 1 package in 55.20s                                                 faiss-cpu            ------------------------------ 17.90 MiB/17.90 MiB         
Installed 1 package in 2ms                                  
 + faiss-cpu==1.15.0


In [25]:
! uv pip install langchain-openai langchain-community faiss-cpu

Using Python 3.13.12 environment at: /home/mohamed-tamer/Downloads/MyProjects/LLMops/.venv
Resolved 56 packages in 1.02s                                        
Prepared 1 package in 204ms                                                  langchain-openai     ------------------------------ 122.15 KiB/122.15 KiB       
Installed 1 package in 3ms.6.0                              
 + langchain-openai==1.6.0


In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()

vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [29]:
docs = vectorstore.similarity_search("What is the main topic?", k=4)

for i, doc in enumerate(docs, start=1):
    print(f"Document {i}: {doc.page_content}")

Document 1: Phase
Description
Perception
The AI system gathers and interprets data from its environment.
Planning
Based on the perceived environment and its goal, the AI formulates a sequence of actions.
Action
The AI executes the planned actions in the real or virtual world.
Learning
The AI evaluates the outcomes of its actions and updates its internal models or strategies to improve future performance.
Document 2: The Future of Agentic AI
As agentic AI continues to evolve, it is expected to bring about significant transformations in how we interact with technology and how complex problems are solved.  Challenges such as ensuring safety, interpretability, and ethical decision-making remain critical areas of research and development for the widespread adoption of agentic AI.
Document 3: * Planning and Reasoning: They can formulate plans to reach their goals and reason about the consequences of their actions.
* Action Execution: They can interact with the environment to implement their 

In [30]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)

Document 1:
Understanding Agentic AI
Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals.  This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.
Key Characteristics of Agentic AI
Agentic AI systems are distinct from traditional AI models due to several core characteristics:
* Goal-Oriented: They possess a clear objective or goal that they strive to achieve.
* Autonomy: They can operate independently, making decisions and taking actions without continuous human oversight.
* Perception: They can interpret information from their environment to inform their decisions.
* Planning and Reasoning: They can formulate plans to reach their goals and reason about the consequences of their actions.
--------------------------------------

In [ ]:
# LangChain 1.0 moved prompts out of the root 'langchain' package.
# In langchain>=1.0, use langchain_core.prompts instead of langchain.prompts.
from langchain_core.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""